## Compute the inter-rater agreement between rater 1 and rater 2

### Setup and Helper Functions

#### Imports

In [1]:
from pathlib import Path

import pandas as pd

#### Datapaths

In [2]:
data_path = Path("../../results")

### Load data

In [3]:
rater_01 = pd.read_csv(data_path.joinpath("combined_data_rater_01.csv"), index_col=[0, 1, 2, 3, 4])
rater_01.index = rater_01.index.droplevel(0)
rater_02 = pd.read_csv(data_path.joinpath("combined_data_rater_02.csv"), index_col=[0, 1, 2, 3, 4])
rater_02.index = rater_02.index.droplevel(0)
intersecting_ids = rater_01.index.intersection(rater_02.index)

### Compute inter-rater agreement


#### Assign cardiac cycle to agreement bins
Agreement is categorized based on the difference between the annotated B-points of rater 1 and rater 2
- Low Agreement: difference > 10 ms
- Medium Agreement: 4 ms <= difference <= 10 ms
- High Agreement: difference < 4 ms

In [4]:
inter_rater_agreement = rater_01.loc[intersecting_ids].copy()
inter_rater_agreement["b_point_sample_reference"] = (
    inter_rater_agreement["b_point_sample_reference"] - rater_02.loc[intersecting_ids]["b_point_sample_reference"]
).abs()
inter_rater_agreement["Agreement"] = "Medium Agreement"
inter_rater_agreement = inter_rater_agreement[["b_point_sample_reference", "Agreement"]]
inter_rater_agreement = inter_rater_agreement.rename(columns={"b_point_sample_reference": "rater_difference"})
inter_rater_agreement.loc[inter_rater_agreement["rater_difference"] <= 4, "Agreement"] = "High Agreement"
inter_rater_agreement.loc[inter_rater_agreement["rater_difference"] > 10, "Agreement"] = "Low Agreement"
inter_rater_agreement

rater_difference  \
participant condition phase         heartbeat_id_reference                     
GDN0005     Dummy     HoldingBreath 0                                    0.0   
                                    1                                    2.0   
                                    3                                    2.0   
                                    4                                    0.0   
                                    5                                    2.0   
...                                                                      ...   
VP_032      tsst      Talk          39                                   2.0   
                                    40                                   8.0   
                                    41                                   7.0   
                                    42                                   1.0   
                                    43                                   7.0   

                                                                   Agreement  
participant condition phase         heartbeat_id_reference                    
GDN0005     Dummy     HoldingBreath 0                         High Agreement  
                                    1                         High Agreement  
                                    3                         High Agreement  
                                    4                         High Agreement  
                                    5                         High Agreement  
...                                                                      ...  
VP_032      tsst      Talk          39                        High Agreement  
                                    40                      Medium Agreement  
                                    41                      Medium Agreement  
                                    42                        High Agreement  
                                    43                      Medium Agreement  

[11278 rows x 2 columns]

#### Count cardiac cycles per agreement bin

In [5]:
high_agreement = inter_rater_agreement.loc[inter_rater_agreement["Agreement"] == "High Agreement", "Agreement"].count()
medium_agreement = inter_rater_agreement.loc[
    inter_rater_agreement["Agreement"] == "Medium Agreement", "Agreement"
].count()
low_agreement = inter_rater_agreement.loc[inter_rater_agreement["Agreement"] == "Low Agreement", "Agreement"].count()
dataset_length = inter_rater_agreement.shape[0]

In [9]:
print(f"Mean absolute rater deviation: {inter_rater_agreement['rater_difference'].mean()}")
print(f"Standard deviation of absolute rater deviation: {inter_rater_agreement['rater_difference'].std()}")

Mean absolute rater deviation: 8.346958680617131
Standard deviation of absolute rater deviation: 14.782407928676305


#### Normalize counts by amount of cardiac cylces

In [12]:
high_agreement / dataset_length

np.float64(0.6606667848909381)

In [13]:
medium_agreement / dataset_length

np.float64(0.17201631494945913)

In [14]:
low_agreement / dataset_length

np.float64(0.16731690015960277)

In [69]:
# inter_rater_agreement.to_csv(data_path.joinpath("inter_rater_agreement.csv"))